<a href="https://colab.research.google.com/github/sriharsha0092/Projects/blob/master/geoglyph_from_sel_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Amazon geoglyphs: detection test, restarted from the selected table

Run top to bottom. Starts from `geoglyphs_selected_v2026-08-18.csv` (the 1,003 positives, called `sel`).

**Important:** the random negatives could not be reproduced from an earlier run, so this notebook saves `allp` (with coordinates) *before* the slow feature extraction. All results here come from this draw of negatives and are reported separately from the earlier run.

**Before running:** put your Google Cloud project ID in the first code cell. Do not commit it.

**AI assistance:** parts of this notebook were built with AI assistance while learning; all results were run and checked by the author.

**Status:** exploratory. The deep-learning section at the end was added after seeing the random-forest results (settings fixed before running).

In [ ]:
import os, time
import numpy as np
import pandas as pd
import ee, geemap

PROJECT = "earthproject-496719"     # <- edit; do not commit your real ID

try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)
print("earth engine ready")

earth engine ready


In [ ]:
# Colab only: upload the input files (skip if they are already in the working folder)
try:
    from google.colab import files
    need = [f for f in ["geoglyphs_selected_v2026-08-18.csv", "geoglyphs_clean_v2026-08-18.csv"] if not os.path.exists(f)]
    if need:
        print("choose (the clean file is optional):", need)
        files.upload()
except ImportError:
    pass
print([f for f in os.listdir(".") if f.startswith("geoglyphs_")])

choose (the clean file is optional): ['geoglyphs_selected_v2026-08-18.csv', 'geoglyphs_clean_v2026-08-18.csv']


Saving geoglyphs_selected_v2026-08-18.csv to geoglyphs_selected_v2026-08-18.csv
['geoglyphs_selected_v2026-08-18.csv']


## 1. Load the selected positives

In [ ]:
sel = pd.read_csv("geoglyphs_selected_v2026-08-18.csv")
print(sel.shape)                      # expect (1003, 11)
print(sel.columns.tolist())

clean_path = "geoglyphs_clean_v2026-08-18.csv"
if os.path.exists(clean_path):
    all_features = pd.read_csv(clean_path)           # 1,474 known features: negatives stay >= 1 km from ALL of them
else:
    all_features = sel.copy()
    print("WARNING: clean table not found; using the 1,003 selected features only. Say this in the README.")
print(len(all_features))

(1003, 12)
['code', 'place', 'lat', 'lon', 'elev', 'source', 'a width', 'b width', 'form', 'b_missing', 'img_date', 'b_filled']
1003


## 2. Earth Engine objects: positives, 2025 dry-season composite, indices

In [ ]:
pip install geojson

In [ ]:
pts = sel[["code", "lat", "lon", "a width", "form"]].rename(columns={"a width": "a_width"})
fc = geemap.df_to_ee(pts, latitude="lat", longitude="lon")
print(len(sel), "positives")

1003 positives


In [ ]:
YEAR, CS_THRESH = 2025, 0.60
START, END = f"{YEAR}-06-01", f"{YEAR}-10-01"        # filterDate's end is exclusive, so this includes 30 Sept
region = fc.geometry().bounds()

csp = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")
s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
      .filterBounds(region).filterDate(START, END)
      .linkCollection(csp, ["cs_cdf"])
      .map(lambda im: im.updateMask(im.select("cs_cdf").gte(CS_THRESH))))

comp = s2.median()

def indices(img):
    r = img.divide(10000)                                    # reflectance 0-1
    ndvi = img.normalizedDifference(["B8", "B4"]).rename("NDVI")
    ndmi = img.normalizedDifference(["B8", "B11"]).rename("NDMI")
    savi = r.expression("1.5*(N-R)/(N+R+0.5)", {"N": r.select("B8"), "R": r.select("B4")}).rename("SAVI")
    bsi  = r.expression("((S+R)-(N+B))/((S+R)+(N+B))",
                        {"S": r.select("B11"), "R": r.select("B4"), "N": r.select("B8"), "B": r.select("B2")}).rename("BSI")
    swir1 = r.select("B11").rename("SWIR1")
    return ee.Image.cat([ndvi, savi, ndmi, bsi, swir1])

feat = indices(comp)


## 3. Draw the negatives and build `allp`, saved immediately

In [ ]:
# ---- settings ----
NDVI_LO, NDVI_HI = 0.356224, 0.776250    # replace with your 5th and 95th percentile from step A
MIN_DIST = 1000                  # metres from ANY known feature (use all_features)
REGION_BUFFER = 20000            # only sample within 20 km of the positives
N_WANT = 6000                     # test with 100 first, then 3000
SEED = 42

all_fc = geemap.df_to_ee(all_features[["code","lat","lon"]], latitude="lat", longitude="lon")
region = fc.geometry().buffer(REGION_BUFFER)

ndvi = feat.select("NDVI")
cleared = ndvi.gte(NDVI_LO).And(ndvi.lte(NDVI_HI))
far = all_fc.distance(5000).unmask(5000).gte(MIN_DIST)
ok = cleared.And(far).selfMask()

neg = ok.sample(region=region, scale=30, numPixels=N_WANT*3, seed=SEED, geometries=True)
neg = neg.randomColumn("r", SEED).sort("r").limit(N_WANT)
neg = neg.map(lambda f: f.set({"lon": f.geometry().coordinates().get(0),
                               "lat": f.geometry().coordinates().get(1)}))
neg_df = geemap.ee_to_df(neg)[["lon","lat"]]
print(len(neg_df))


6000


In [ ]:
# borrow widths from real geoglyphs
neg_df["a_width"] = sel["a width"].sample(len(neg_df), replace=True, random_state=SEED).values
neg_df["label"] = 0
neg_df["id"] = ["neg" + str(i) for i in range(len(neg_df))]

# positives
pos = sel[["code","lat","lon","a width"]].rename(columns={"code":"id","a width":"a_width"})
pos["label"] = 1

# everything in one table
allp = pd.concat([pos, neg_df[["id","lat","lon","a_width","label"]]], ignore_index=True)

print(len(allp))
print(allp.label.value_counts())
print(allp.groupby("label")["a_width"].describe())


7003
label
0    6000
1    1003
Name: count, dtype: int64
        count        mean        std   min    25%    50%    75%    max
label                                                                 
0      6000.0  177.874500  69.690565  90.0  126.0  164.0  212.0  604.0
1      1003.0  177.605184  69.791489  90.0  128.0  161.0  210.0  604.0


In [ ]:
# the negatives cannot be regenerated identically, so save them NOW
allp.to_csv("allp_v2_with_coords.csv", index=False)
print("saved allp_v2_with_coords.csv", allp.shape)

saved allp_v2_with_coords.csv (7003, 5)


In [ ]:
cols = ["NDVI","NDMI","BSI","SWIR1"]
bands = feat.select(cols)

def inside(f):
    return f.buffer(ee.Number(f.get("a_width")).divide(2))

def ring(f):
    a = ee.Number(f.get("a_width"))
    return f.buffer(a.multiply(1.5)).difference(f.buffer(a.multiply(0.75)))

def extract(df_in):
    pts = geemap.df_to_ee(df_in, latitude="lat", longitude="lon")
    mean_in   = bands.reduceRegions(collection=pts.map(inside), reducer=ee.Reducer.mean(), scale=10)
    mean_ring = bands.reduceRegions(collection=pts.map(ring),   reducer=ee.Reducer.mean(), scale=10)
    sd_in     = feat.select("NDVI").reduceRegions(collection=pts.map(inside), reducer=ee.Reducer.stdDev(), scale=10)
    inn = geemap.ee_to_df(mean_in)[["id","label"]+cols].rename(columns={c: c+"_in" for c in cols})
    rng = geemap.ee_to_df(mean_ring)[["id"]+cols].rename(columns={c: c+"_ring" for c in cols})
    sd  = geemap.ee_to_df(sd_in)[["id","stdDev"]].rename(columns={"stdDev":"NDVI_sd_in"})
    X = inn.merge(rng, on="id").merge(sd, on="id")
    for c in cols:
        X[c+"_diff"] = X[c+"_in"] - X[c+"_ring"]
    return X


## 4. Extract features, resumable (saved after every batch)

In [ ]:
parts_file = "features_v2_partial.csv"
done = pd.read_csv(parts_file) if os.path.exists(parts_file) else pd.DataFrame()
start = len(done)
print("resuming from row", start)
for i in range(start, len(allp), 250):
    part = extract(allp.iloc[i:i+250])
    done = pd.concat([done, part], ignore_index=True)
    done.to_csv(parts_file, index=False)            # saved after every batch
    print(i, end=" ")

X_all = done.copy()
X_all.to_csv("features_v2.csv", index=False)
print()
print(len(X_all), len(allp), X_all.label.value_counts().to_dict(), int(X_all.isna().sum().sum()))
assert set(X_all["id"]) == set(allp["id"]), "ids differ"

resuming from row 0
0 250 500 750 1000 1250 1500 1750 2000 2250 2500 2750 3000 3250 3500 3750 4000 4250 4500 4750 5000 5250 5500 5750 6000 6250 6500 6750 7000 
7003 7003 {0: 6000, 1: 1003} 0


## 5. Matching, random forest and spatial validation (your original cells)

In [ ]:
from sklearn.preprocessing import StandardScaler

# BEFORE: gap between groups with the whole pool
check = ["NDVI_ring","SWIR1_ring","BSI_ring","NDMI_ring"]
print("before matching")
print(X_all.groupby("label")[check].mean())

# match on the surroundings only
mcols = ["NDVI_ring","SWIR1_ring"]
P = X_all[X_all.label==1].reset_index(drop=True)
N = X_all[X_all.label==0].reset_index(drop=True)

sc = StandardScaler().fit(X_all[mcols])
Pz, Nz = sc.transform(P[mcols]), sc.transform(N[mcols])

used = np.zeros(len(N), dtype=bool)
chosen = []
rng = np.random.default_rng(42)
for i in rng.permutation(len(P)):
    d = np.sqrt(((Nz - Pz[i])**2).sum(axis=1))
    d[used] = np.inf
    j = d.argmin()
    used[j] = True
    chosen.append(j)

M = pd.concat([P, N.iloc[chosen]], ignore_index=True)

# AFTER
print("after matching")
print(M.groupby("label")[check].mean())
print(M.label.value_counts())
M.to_csv("features_v2_matched.csv", index=False)


before matching
       NDVI_ring  SWIR1_ring  BSI_ring  NDMI_ring
label                                            
0       0.598403    0.260660  0.007558   0.026500
1       0.566934    0.295264  0.043332  -0.008376
after matching
       NDVI_ring  SWIR1_ring  BSI_ring  NDMI_ring
label                                            
0       0.567442    0.294887  0.044072  -0.010261
1       0.566934    0.295264  0.043332  -0.008376
label
1    1003
0    1003
Name: count, dtype: int64


In [ ]:
M = M.merge(allp[["id","lat","lon","a_width"]], on="id")
print(len(M))   # 2006

feat_cols = ["NDVI_in","NDMI_in","BSI_in","SWIR1_in","NDVI_sd_in",
             "NDVI_diff","NDMI_diff","BSI_diff","SWIR1_diff"]
print(M.groupby("label")[feat_cols].mean().T)


2006
label              0         1
NDVI_in     0.551609  0.543884
NDMI_in    -0.028351 -0.037539
BSI_in      0.062412  0.072602
SWIR1_in    0.302104  0.311502
NDVI_sd_in  0.068139  0.055278
NDVI_diff  -0.015832 -0.023050
NDMI_diff  -0.018089 -0.029162
BSI_diff    0.018341  0.029270
SWIR1_diff  0.007217  0.016238


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, confusion_matrix

feat_cols = ["NDVI_in","NDMI_in","BSI_in","SWIR1_in","NDVI_sd_in",
             "NDVI_diff","NDMI_diff","BSI_diff","SWIR1_diff"]
y = M["label"]
Xtr, Xte, ytr, yte = train_test_split(M, y, test_size=0.3, random_state=42, stratify=y)

def report(name, model, cols):
    model.fit(Xtr[cols], ytr)
    p = model.predict_proba(Xte[cols])[:, 1]
    pred = (p >= 0.5).astype(int)          # cutoff fixed at 0.5 (classes are balanced)
    print(name)
    print("  AUC      ", round(roc_auc_score(yte, p), 3))
    print("  accuracy ", round(accuracy_score(yte, pred), 3))
    print("  precision", round(precision_score(yte, pred), 3))
    print("  recall   ", round(recall_score(yte, pred), 3))
    print(confusion_matrix(yte, pred))

report("baseline: NDVI_in only", make_pipeline(StandardScaler(), LogisticRegression()), ["NDVI_in"])
report("logistic, 9 features",   make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), feat_cols)
report("random forest, 9 features", RandomForestClassifier(n_estimators=300, random_state=42), feat_cols)


baseline: NDVI_in only
  AUC       0.545
  accuracy  0.533
  precision 0.534
  recall    0.522
[[164 137]
 [144 157]]
logistic, 9 features
  AUC       0.615
  accuracy  0.565
  precision 0.561
  recall    0.595
[[161 140]
 [122 179]]
random forest, 9 features
  AUC       0.636
  accuracy  0.59
  precision 0.584
  recall    0.621
[[168 133]
 [114 187]]


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

BLOCK = 0.25      # degrees, about 28 km -- fix this before running
M["block"] = (np.floor(M["lat"]/BLOCK).astype(int).astype(str) + "_" +
              np.floor(M["lon"]/BLOCK).astype(int).astype(str))
print(M["block"].nunique(), "blocks")

models = {
  "baseline NDVI_in": (make_pipeline(StandardScaler(), LogisticRegression()), ["NDVI_in"]),
  "logistic":         (make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), feat_cols),
  "random forest":    (RandomForestClassifier(n_estimators=300, random_state=42), feat_cols),
}

gss = GroupShuffleSplit(n_splits=20, test_size=0.3, random_state=42)
res = {k: [] for k in models}
for tr, te in gss.split(M, M["label"], groups=M["block"]):
    ytr_, yte_ = M["label"].iloc[tr], M["label"].iloc[te]
    if yte_.nunique() < 2:
        continue
    for k, (m, cols) in models.items():
        m.fit(M[cols].iloc[tr], ytr_)
        res[k].append(roc_auc_score(yte_, m.predict_proba(M[cols].iloc[te])[:, 1]))

for k, v in res.items():
    print(k, "mean AUC", round(np.mean(v), 3), "range", round(min(v), 3), "-", round(max(v), 3), "n splits", len(v))


152 blocks
baseline NDVI_in mean AUC 0.514 range 0.423 - 0.555 n splits 20
logistic mean AUC 0.613 range 0.586 - 0.644 n splits 20
random forest mean AUC 0.619 range 0.583 - 0.648 n splits 20


In [ ]:
def spatial_auc(D, name="random forest", n=20):
    m, cols = models[name]
    out = []
    for tr, te in GroupShuffleSplit(n_splits=n, test_size=0.3, random_state=42).split(D, D["label"], groups=D["block"]):
        if D["label"].iloc[te].nunique() < 2 or D["label"].iloc[tr].nunique() < 2:
            continue
        m.fit(D[cols].iloc[tr], D["label"].iloc[tr])
        out.append(roc_auc_score(D["label"].iloc[te], m.predict_proba(D[cols].iloc[te])[:, 1]))
    return round(np.mean(out), 3), round(min(out), 3), round(max(out), 3), len(out)

print("all rows         ", spatial_auc(M))

filled_ids = sel.loc[sel["b_filled"], "code"]          # rows where b was assumed = a
print("without b_filled ", spatial_auc(M[~M["id"].isin(filled_ids)]))

P2 = M.copy()
P2["label"] = np.random.default_rng(0).permutation(P2["label"].values)
print("shuffled labels  ", spatial_auc(P2))


all rows          (np.float64(0.619), np.float64(0.583), np.float64(0.648), 20)
without b_filled  (np.float64(0.637), np.float64(0.572), np.float64(0.689), 20)
shuffled labels   (np.float64(0.502), np.float64(0.471), np.float64(0.537), 20)


In [ ]:
M.to_csv("features_v2_matched_with_coords.csv", index=False)

## 6. Exploratory addition: small CNN on image patches (added after seeing the RF results)

Settings fixed before running: 33x33 pixel patches at 20 m, bands B4/B8/B11, 20 epochs, no early stopping, same matched data and same spatial blocks and seeds as the random forest. A shuffled-label control is included. Report whatever comes out.

In [ ]:
R = 16                                   # patch is (2R+1) = 33 pixels per side
SCALE_CHIP = 20                          # metres per pixel
BANDS_CHIP = ["B4", "B8", "B11"]

img = comp.select(BANDS_CHIP).divide(10000).unmask(0)
patch_img = img.neighborhoodToArray(ee.Kernel.square(R, "pixels"))

def get_chips(df, batch=40):
    X, ids = [], []
    for i in range(0, len(df), batch):
        part = df.iloc[i:i+batch]
        pts_ = geemap.df_to_ee(part[["id", "lat", "lon"]], latitude="lat", longitude="lon")
        for attempt in range(3):
            try:
                out = patch_img.sampleRegions(collection=pts_, properties=["id"],
                                              scale=SCALE_CHIP, geometries=False).getInfo()
                break
            except Exception as e:
                print("retry", i, e); time.sleep(5)
        for f in out["features"]:
            p = f["properties"]
            X.append(np.stack([np.array(p[b], dtype="float32") for b in BANDS_CHIP], axis=-1))
            ids.append(p["id"])
        print(i, end=" ")
    return np.array(X), ids

chips, chip_ids = get_chips(M)
print(chips.shape)
np.save("chips_v1.npy", chips)
pd.Series(chip_ids).to_csv("chip_ids_v1.csv", index=False)

0 40 80 120 160 200 240 280 320 360 400 

440 

480 520 560 600 640 680 720 760 800 840 880 920 960 1000 1040 1080 1120 1160 1200 1240 1280 1320 1360 1400 1440 1480 1520 1560 1600 1640 1680 1720 1760 1800 1840 1880 1920 1960 2000 (2006, 33, 33, 3)


In [ ]:
idx = {k: i for i, k in enumerate(chip_ids)}
Mc = M[M["id"].isin(idx)].reset_index(drop=True)
Xc = np.stack([chips[idx[k]] for k in Mc["id"]])
yc = Mc["label"].values
print(len(M), len(Mc), Xc.shape)         # if len(Mc) < len(M), say so in the README

2006 2006 (2006, 33, 33, 3)


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models as kmodels
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

def make_cnn():
    m = kmodels.Sequential([
        layers.Input((2*R+1, 2*R+1, 3)),
        layers.RandomFlip("horizontal_and_vertical"),
        layers.Conv2D(16, 3, activation="relu"), layers.BatchNormalization(), layers.MaxPooling2D(),
        layers.Conv2D(32, 3, activation="relu"), layers.BatchNormalization(), layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu"), layers.GlobalAveragePooling2D(),
        layers.Dropout(0.5), layers.Dense(1, activation="sigmoid")])
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy")
    return m

def cnn_spatial_auc(X, y, groups, n_splits=20, epochs=20, shuffle_labels=False):
    tf.keras.utils.set_random_seed(42)
    y = np.random.default_rng(0).permutation(y) if shuffle_labels else y
    out = []
    for tr, te in GroupShuffleSplit(n_splits=n_splits, test_size=0.3, random_state=42).split(X, y, groups=groups):
        if len(set(y[te])) < 2 or len(set(y[tr])) < 2:
            continue
        mu, sd = X[tr].mean((0, 1, 2)), X[tr].std((0, 1, 2)) + 1e-6      # train-fold stats only
        m = make_cnn()
        m.fit((X[tr]-mu)/sd, y[tr], epochs=epochs, batch_size=64, verbose=0)
        out.append(roc_auc_score(y[te], m.predict((X[te]-mu)/sd, verbose=0).ravel()))
    return round(np.mean(out), 3), round(min(out), 3), round(max(out), 3), len(out)

print("CNN spatial AUC     ", cnn_spatial_auc(Xc, yc, Mc["block"].values))
print("CNN shuffled labels ", cnn_spatial_auc(Xc, yc, Mc["block"].values, n_splits=5, shuffle_labels=True))
print("compare: random forest spatial AUC 0.636 (0.575-0.681)")

CNN spatial AUC      (np.float64(0.644), np.float64(0.602), np.float64(0.68), 20)
CNN shuffled labels  (np.float64(0.501), np.float64(0.489), np.float64(0.519), 5)
compare: random forest spatial AUC 0.636 (0.575-0.681)


In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

def spatial_auc_cols(D, cols, n=20):
    out = []
    for tr, te in GroupShuffleSplit(n_splits=n, test_size=0.3, random_state=42).split(D, D["label"], groups=D["block"]):
        if D["label"].iloc[te].nunique() < 2 or D["label"].iloc[tr].nunique() < 2:
            continue
        m = RandomForestClassifier(n_estimators=300, random_state=42).fit(D[cols].iloc[tr], D["label"].iloc[tr])
        out.append(roc_auc_score(D["label"].iloc[te], m.predict_proba(D[cols].iloc[te])[:, 1]))
    return round(np.mean(out), 3), round(min(out), 3), round(max(out), 3), len(out)

_, edges = pd.qcut(M.loc[M.label == 1, "a_width"], 3, retbins=True, duplicates="drop")
M["wbin"] = pd.cut(M["a_width"], edges, include_lowest=True)
for b, D in M.groupby("wbin", observed=True):
    print(b, "| pos", int(D.label.sum()), "neg", int((D.label == 0).sum()), "|", spatial_auc_cols(D, feat_cols))

(89.999, 137.0] | pos 335 neg 361 | (np.float64(0.574), np.float64(0.535), np.float64(0.63), 20)
(137.0, 192.0] | pos 337 neg 317 | (np.float64(0.586), np.float64(0.506), np.float64(0.661), 20)
(192.0, 604.0] | pos 331 neg 325 | (np.float64(0.642), np.float64(0.553), np.float64(0.698), 20)


In [ ]:
WET_START, WET_END = "2024-12-01", "2025-04-01"      # wet season before your 2025 dry composite

s2_wet = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
          .filterBounds(region).filterDate(WET_START, WET_END)
          .linkCollection(csp, ["cs_cdf"])
          .map(lambda im: im.updateMask(im.select("cs_cdf").gte(CS_THRESH))))
feat_wet = indices(s2_wet.median())

def extract_from(df_in, feat_img):
    idx_cols = ["NDVI", "NDMI", "BSI", "SWIR1"]          # local, so a global `cols` can't break it
    b = feat_img.select(idx_cols)
    pts = geemap.df_to_ee(df_in, latitude="lat", longitude="lon")
    mean_in   = b.reduceRegions(collection=pts.map(inside), reducer=ee.Reducer.mean(), scale=10)
    mean_ring = b.reduceRegions(collection=pts.map(ring),   reducer=ee.Reducer.mean(), scale=10)
    sd_in     = feat_img.select("NDVI").reduceRegions(collection=pts.map(inside), reducer=ee.Reducer.stdDev(), scale=10)
    inn  = geemap.ee_to_df(mean_in)[["id","label"]+idx_cols].rename(columns={c: c+"_in" for c in idx_cols})
    rng_ = geemap.ee_to_df(mean_ring)[["id"]+idx_cols].rename(columns={c: c+"_ring" for c in idx_cols})
    sd   = geemap.ee_to_df(sd_in)[["id","stdDev"]].rename(columns={"stdDev":"NDVI_sd_in"})
    X = inn.merge(rng_, on="id").merge(sd, on="id")
    for c in idx_cols:
        X[c+"_diff"] = X[c+"_in"] - X[c+"_ring"]
    return X

base = M[["id","lat","lon","a_width","label"]]
pf = "features_wet_partial.csv"
wet = pd.read_csv(pf) if os.path.exists(pf) else pd.DataFrame()
for i in range(len(wet), len(base), 250):
    wet = pd.concat([wet, extract_from(base.iloc[i:i+250], feat_wet)], ignore_index=True)
    wet.to_csv(pf, index=False); print(i, end=" ")

wet_cols = [c + "_w" for c in feat_cols]
W = wet.drop(columns="label").rename(columns={c: c + "_w" for c in feat_cols})
Mw = M.merge(W, on="id", how="left")
print(Mw[wet_cols].isna().mean().round(2))            # share of rows with no clear wet-season pixels
D = Mw.dropna(subset=wet_cols)
print(len(M), "->", len(D), "rows")
print("dry only  ", spatial_auc_cols(D, feat_cols))
print("dry + wet ", spatial_auc_cols(D, feat_cols + wet_cols))
print("wet only  ", spatial_auc_cols(D, wet_cols))

0 250 500 750 1000 1250 1500 1750 2000 NDVI_in_w       0.0
NDMI_in_w       0.0
BSI_in_w        0.0
SWIR1_in_w      0.0
NDVI_sd_in_w    0.0
NDVI_diff_w     0.0
NDMI_diff_w     0.0
BSI_diff_w      0.0
SWIR1_diff_w    0.0
dtype: float64
2006 -> 2006 rows
dry only   (np.float64(0.619), np.float64(0.583), np.float64(0.648), 20)
dry + wet  (np.float64(0.653), np.float64(0.617), np.float64(0.692), 20)
wet only   (np.float64(0.58), np.float64(0.535), np.float64(0.621), 20)


In [ ]:
# ---- generic extractor (own band list, so the global `cols` trap can't bite) ----
def extract_bands(df_in, img, names, scale=10):
    b = img.select(names)
    pts = geemap.df_to_ee(df_in, latitude="lat", longitude="lon")
    m_in = geemap.ee_to_df(b.reduceRegions(collection=pts.map(inside), reducer=ee.Reducer.mean(), scale=scale))[["id"] + names]
    m_rg = geemap.ee_to_df(b.reduceRegions(collection=pts.map(ring),   reducer=ee.Reducer.mean(), scale=scale))[["id"] + names]
    X = m_in.rename(columns={c: c + "_in" for c in names}).merge(
        m_rg.rename(columns={c: c + "_ring" for c in names}), on="id")
    for c in names:
        X[c + "_diff"] = X[c + "_in"] - X[c + "_ring"]
    return X

def run_batches(tag, img, names, scale=10, step=250):
    pf = f"features_{tag}_partial.csv"
    out = pd.read_csv(pf) if os.path.exists(pf) else pd.DataFrame()
    for i in range(len(out), len(base), step):
        out = pd.concat([out, extract_bands(base.iloc[i:i+step], img, names, scale)], ignore_index=True)
        out.to_csv(pf, index=False); print(i, end=" ")
    return out

terrain = elev.addBands(ee.Terrain.slope(elev).rename("slope"))
T = run_batches("terrain", terrain, ["elev", "slope"], scale=30, step=100)# ---- terrain (fast) ----
elev = ee.Image("USGS/SRTMGL1_003").select("elevation").rename("elev")
terrain = (elev.addBands(ee.Terrain.slope(elev).rename("slope"))
               .addBands(elev.subtract(elev.focalMean(1000, "circle", "meters")).rename("relief")))
T = run_batches("terrain", terrain, ["elev", "slope", "relief"], scale=30)

# ---- Sentinel-1 radar, same dry season (dB values; VV-VH in dB is the cross-pol ratio) ----
s1 = (ee.ImageCollection("COPERNICUS/S1_GRD")
      .filterBounds(region).filterDate("2025-06-01", "2025-10-01")
      .filter(ee.Filter.eq("instrumentMode", "IW"))
      .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
      .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH")))
print("S1 scenes:", s1.size().getInfo())                 # if this is 0 or tiny, stop and tell me
s1m = s1.select(["VV", "VH"]).median()
s1_feat = s1m.addBands(s1m.select("VV").subtract(s1m.select("VH")).rename("VVminusVH"))
S = run_batches("s1", s1_feat, ["VV", "VH", "VVminusVH"])

# ---- compare, all on the same rows ----
def add(Mx, tbl, tag):
    t_ = tbl.drop(columns=[c for c in ["label"] if c in tbl.columns]).rename(
        columns={c: c + "_" + tag for c in tbl.columns if c not in ("id", "label")})
    return Mx.merge(t_, on="id", how="left"), [c for c in t_.columns if c != "id"]

Mx, t_cols = add(M, T, "t")
Mx, s_cols = add(Mx, S, "s1")
D = Mx.dropna(subset=t_cols + s_cols)
print(len(M), "->", len(D), "rows")
for name, cs in [("optical dry only", feat_cols),
                 ("terrain only    ", t_cols),
                 ("radar only      ", s_cols),
                 ("optical + radar ", feat_cols + s_cols),
                 ("optical + terrain", feat_cols + t_cols),
                 ("all three       ", feat_cols + s_cols + t_cols)]:
    print(name, spatial_auc_cols(D, cs))

0 100 200 300 400 500 600 700 800 900 1000 1100 1200 1300 1400 1500 1600 1700 1800 1900 2000 S1 scenes: 415
0 250 500 750 1000 1250 1500 1750 2000 2006 -> 2006 rows
optical dry only (np.float64(0.619), np.float64(0.583), np.float64(0.648), 20)
terrain only     (np.float64(0.662), np.float64(0.609), np.float64(0.7), 20)
radar only       (np.float64(0.548), np.float64(0.495), np.float64(0.587), 20)
optical + radar  (np.float64(0.641), np.float64(0.606), np.float64(0.677), 20)
optical + terrain (np.float64(0.716), np.float64(0.683), np.float64(0.751), 20)
all three        (np.float64(0.724), np.float64(0.695), np.float64(0.768), 20)


In [ ]:
D2 = D.copy()
D2["label"] = np.random.default_rng(0).permutation(D2["label"].values)
print("shuffled, optical+terrain:", spatial_auc_cols(D2, feat_cols + t_cols, n=5))   # should be ~0.50

for c in ["elev", "slope"]:
    print(c, "only:", spatial_auc_cols(D, [x for x in t_cols if x.startswith(c)]))

shuffled, optical+terrain: (np.float64(0.507), np.float64(0.486), np.float64(0.524), 5)
elev only: (np.float64(0.615), np.float64(0.581), np.float64(0.646), 20)
slope only: (np.float64(0.555), np.float64(0.514), np.float64(0.591), 20)
